### Tratamiento y Preparación de Datos para Modelos Supervisados

Para garantizar que el ecosistema de algoritmos funcione bajo condiciones óptimas y sin violar sus supuestos matemáticos, se ha diseñado un pipeline de preprocesamiento riguroso:

1.  **Mitigación de Sesgo por Submuestreo:** Redujimos el dominio masivo del Yaguareté fijando un tope máximo de 3,500 muestras por clase, previniendo que los clasificadores colapsen hacia la clase mayoritaria.
2.  **Transformación Cíclica del Tiempo:** Convertimos el mes lineal (1-12) en funciones trigonométricas sinusoidales y cosenoidales, capturando la naturaleza continua y estacional del tiempo para modelos geométricos.
3.  **Prevención de Multicolinealidad:** Al aplicar *One-Hot Encoding*, activamos el parámetro `drop_first=True`. Esto elimina la redundancia lineal estricta entre columnas binarias, un requisito indispensable para la estabilidad de la Regresión Logística y el Análisis Discriminante Lineal (LDA).
4.  **Aislamiento de Escalado (Data Leakage):** El ajuste de la estandarización (`StandardScaler`) se realizó de forma estricta únicamente sobre la partición de entrenamiento, asegurando una validación externa legítima en el set de prueba.


Carga, Filtrado de Especies y Balanceo de Clases

In [5]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Cargar el dataset
RUTA_DATOS = os.path.join('..', 'Datos', 'Felinos_Sudamerica.csv')
df = pd.read_csv(RUTA_DATOS, sep='\t', low_memory=False)

# Definir las 6 especies nativas principales para el modelo multiclase
especies_target = [
    'Panthera onca', 'Puma concolor', 'Leopardus pardalis', 
    'Leopardus guigna', 'Herpailurus yagouaroundi', 'Leopardus geoffroyi'
]

# Filtrar por especies objetivo y eliminar nulos iniciales en variables críticas
df_fil = df[df['species'].isin(especies_target)].dropna(subset=['decimalLatitude', 'decimalLongitude', 'month', 'year'])

# Aplicar Submuestreo (Undersampling) para balancear las clases
TOPE_REGISTROS = 3500
df_balanceado = pd.DataFrame()

np.random.seed(42) # Fijamos semilla para reproducibilidad 

for esp in especies_target:
    df_esp = df_fil[df_fil['species'] == esp]
    if len(df_esp) > TOPE_REGISTROS:
        df_esp = df_esp.sample(n=TOPE_REGISTROS, random_state=42)
    df_balanceado = pd.concat([df_balanceado, df_esp], axis=0)

print("=== DATASET FILTRADO Y BALANCEADO ===")
print(df_balanceado['species'].value_counts())


=== DATASET FILTRADO Y BALANCEADO ===
species
Panthera onca               3500
Puma concolor               3500
Leopardus pardalis          3500
Leopardus guigna            3449
Herpailurus yagouaroundi    2138
Leopardus geoffroyi         1123
Name: count, dtype: int64


Ingeniería de Características Cíclicas

In [6]:
# Mapeo cíclico del mes a coordenadas en un círculo
df_balanceado['sin_month'] = np.sin(2 * np.pi * df_balanceado['month'] / 12.0)
df_balanceado['cos_month'] = np.cos(2 * np.pi * df_balanceado['month'] / 12.0)

# Verificamos las nuevas columnas descriptivas
print("\nVariables temporales transformadas:")
display(df_balanceado[['month', 'sin_month', 'cos_month']].head(3))



Variables temporales transformadas:


,month,sin_month,cos_month
101031,11.0,-5.000000e-01,0.866025
92597,11.0,-5.000000e-01,0.866025
115760,6.0,1.224647e-16,-1.000000


Codificación de Variables Categóricas (One-Hot Encoding)

In [7]:
# Selección de variables para el espacio de características (X) y la etiqueta (y)
features_categoricas = ['countryCode', 'basisOfRecord']
features_numericas = ['decimalLatitude', 'decimalLongitude', 'year', 'sin_month', 'cos_month']

# Generar variables dummies (One-Hot Encoding) evitando la colinealidad (drop_first=True)
# drop_first=True es vital para Regresión Logística y LDA, ya que evita la trampa de la variable ficticia
X_cat = pd.get_dummies(df_balanceado[features_categoricas], drop_first=True, dtype=int)
X_num = df_balanceado[features_numericas].copy()

# Unificar todo el set de características X
X = pd.concat([X_num, X_cat], axis=1)

# Codificar la variable objetivo (y) de texto a etiquetas numéricas (0 a 5)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df_balanceado['species'])

print(f"\nMatriz de Características (X): {X.shape[0]} muestras, {X.shape[1]} columnas.")
print("Especies codificadas numéricamente:")
for i, clase in enumerate(le.classes_):
    print(f"Clase {i} -> {clase}")



Matriz de Características (X): 17210 muestras, 17 columnas.
Especies codificadas numéricamente:
Clase 0 -> Herpailurus yagouaroundi
Clase 1 -> Leopardus geoffroyi
Clase 2 -> Leopardus guigna
Clase 3 -> Leopardus pardalis
Clase 4 -> Panthera onca
Clase 5 -> Puma concolor


División y Escalado de Datos (StandardScaler)

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# División en Train (80%) y Test (20%) estratificado por el desbalance remanente
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Escalado Estándar (Media = 0, Desviación Estándar = 1)
scaler = StandardScaler()

# Ajustamos y transformamos el conjunto de entrenamiento
X_train_scaled = scaler.fit_transform(X_train)

# Transformamos el conjunto de prueba usando los parámetros del entrenamiento (Previene Data Leakage)
X_test_scaled = scaler.transform(X_test)

# Convertimos a DataFrame para mantener el orden visual en VS Code
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("\n=== PROCESAMIENTO COMPLETADO CON ÉXITO ===")
print(f"X_train_scaled para algoritmos lineales/distancias: {X_train_scaled.shape}")
print(f"X_train sin escalar (para Árboles de Decisión): {X_train.shape}")



=== PROCESAMIENTO COMPLETADO CON ÉXITO ===
X_train_scaled para algoritmos lineales/distancias: (13768, 17)
X_train sin escalar (para Árboles de Decisión): (13768, 17)


In [9]:
# Guardar los conjuntos de datos en la carpeta Datos
X_train_scaled.to_csv('../Datos/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../Datos/X_test_scaled.csv', index=False)
X_train.to_csv('../Datos/X_train_raw.csv', index=False)
X_test.to_csv('../Datos/X_test_raw.csv', index=False)

pd.DataFrame(y_train, columns=['target']).to_csv('../Datos/y_train.csv', index=False)
pd.DataFrame(y_test, columns=['target']).to_csv('../Datos/y_test.csv', index=False)

print("¡Matrices de entrenamiento y prueba exportadas con éxito!")


¡Matrices de entrenamiento y prueba exportadas con éxito!
